# Notebook 02: Vector Stores with ChromaDB

In Notebook 01, we re-computed embeddings every time we searched. In a real system, you have thousands (or millions) of documents — re-embedding all of them on every query would be way too slow.

That's where **vector stores** come in.

## What is a Vector Store?

A vector store (or vector database) is a database designed to:
1. **Store** high-dimensional vectors (embeddings)
2. **Index** them for fast approximate nearest-neighbor search
3. **Retrieve** the most similar vectors to a query efficiently

Traditional databases do exact lookups (`WHERE id = 42`). Vector stores do similarity lookups (`find the 5 vectors closest to this query vector`).

## Why ChromaDB?

ChromaDB is a great choice for learning because it:
- Runs locally (no cloud account needed)
- Has a simple Python API
- Persists data to disk
- Handles embedding automatically (or lets you bring your own)

## In this notebook you will:
1. Set up a ChromaDB collection
2. Add documents with metadata
3. Query by semantic similarity
4. Filter by metadata
5. Update and delete documents

## Step 1: Setup

In [ ]:
# !pip install chromadb sentence-transformers

import chromadb
from chromadb.utils import embedding_functions
import json

print(f"ChromaDB version: {chromadb.__version__}")

## Step 2: Create a ChromaDB Client and Collection

A **collection** in ChromaDB is like a table — it holds documents, their embeddings, and metadata.

In [ ]:
# Use sentence-transformers as our embedding function
# ChromaDB will call this automatically when we add or query documents
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create a persistent client (saves to disk)
client = chromadb.PersistentClient(path="../data/chroma_db")

# Create (or get existing) collection
# delete_collection first if you want a fresh start
try:
    client.delete_collection("ai_concepts")
except:
    pass

collection = client.create_collection(
    name="ai_concepts",
    embedding_function=embedding_fn,
    metadata={"description": "AI and ML concepts knowledge base"}
)

print(f"Collection '{collection.name}' created")
print(f"Document count: {collection.count()}")

## Step 3: Add Documents

Each document needs:
- `id`: A unique string identifier
- `document`: The text content
- `metadata` (optional): Key-value pairs for filtering

In [ ]:
documents = [
    "Machine learning is a subset of AI that enables computers to learn from data without explicit programming.",
    "Supervised learning uses labeled training data where the correct output is known for each input.",
    "Unsupervised learning finds patterns in data without labeled examples, such as clustering.",
    "Reinforcement learning trains agents through rewards and penalties rather than labeled data.",
    "Neural networks consist of layers of interconnected nodes that transform data through learned weights.",
    "The Transformer architecture uses self-attention mechanisms and powers most modern LLMs.",
    "Large language models are trained on massive text datasets with billions of parameters.",
    "RAG (Retrieval-Augmented Generation) combines a retriever with a language model to ground answers in documents.",
    "Vector embeddings represent text as numerical vectors where similar meanings are geometrically close.",
    "Fine-tuning adapts a pre-trained model to a specific task by continuing training on domain data.",
]

ids = [f"doc_{i}" for i in range(len(documents))]

metadatas = [
    {"topic": "ml_basics", "difficulty": "beginner"},
    {"topic": "ml_basics", "difficulty": "beginner"},
    {"topic": "ml_basics", "difficulty": "beginner"},
    {"topic": "ml_basics", "difficulty": "beginner"},
    {"topic": "deep_learning", "difficulty": "intermediate"},
    {"topic": "deep_learning", "difficulty": "intermediate"},
    {"topic": "llms", "difficulty": "intermediate"},
    {"topic": "rag", "difficulty": "advanced"},
    {"topic": "embeddings", "difficulty": "intermediate"},
    {"topic": "llms", "difficulty": "advanced"},
]

# Add all documents — ChromaDB embeds them automatically
collection.add(
    documents=documents,
    ids=ids,
    metadatas=metadatas
)

print(f"Added {len(documents)} documents")
print(f"Collection now has {collection.count()} documents")

## Step 4: Query by Semantic Similarity

In [ ]:
def pretty_query(collection, query_text, n_results=3, where=None):
    """Run a query and print results in a readable format."""
    kwargs = {"query_texts": [query_text], "n_results": n_results}
    if where:
        kwargs["where"] = where
    
    results = collection.query(**kwargs)
    
    print(f"Query: '{query_text}'")
    if where:
        print(f"Filter: {where}")
    print("-" * 60)
    
    for i, (doc, dist, meta) in enumerate(zip(
        results['documents'][0],
        results['distances'][0],
        results['metadatas'][0]
    )):
        # ChromaDB returns L2 distance by default; convert to similarity score
        score = 1 / (1 + dist)
        print(f"  {i+1}. [score: {score:.3f}] [{meta['topic']}] {doc[:80]}...")
    print()

In [ ]:
# Basic semantic search
pretty_query(collection, "How do transformers and attention work?")
pretty_query(collection, "What is the difference between supervised and unsupervised learning?")
pretty_query(collection, "How can I add domain knowledge to an LLM?")

## Step 5: Metadata Filtering

Metadata filters let you combine semantic search with structured filtering — like SQL WHERE clauses but alongside vector search.

In [ ]:
# Only return beginner-level documents
pretty_query(
    collection,
    "How does learning from data work?",
    n_results=3,
    where={"difficulty": "beginner"}
)

# Only return documents about RAG or embeddings
pretty_query(
    collection,
    "How does retrieval work?",
    n_results=2,
    where={"topic": {"$in": ["rag", "embeddings"]}}
)

## Step 6: Update and Delete Documents

In [ ]:
# Peek at a document before update
before = collection.get(ids=["doc_7"])
print("Before update:")
print(f"  {before['documents'][0]}")
print(f"  metadata: {before['metadatas'][0]}")

# Update a document
collection.update(
    ids=["doc_7"],
    documents=["RAG (Retrieval-Augmented Generation) retrieves relevant documents from a vector store and passes them as context to an LLM to ground the generated response in factual information."],
    metadatas=[{"topic": "rag", "difficulty": "advanced", "updated": True}]
)

after = collection.get(ids=["doc_7"])
print("\nAfter update:")
print(f"  {after['documents'][0]}")
print(f"  metadata: {after['metadatas'][0]}")

In [ ]:
# Add a new document
collection.add(
    documents=["Prompt engineering is the practice of crafting inputs to guide LLM behavior without changing model weights."],
    ids=["doc_10"],
    metadatas=[{"topic": "llms", "difficulty": "beginner"}]
)
print(f"Collection now has {collection.count()} documents")

# Delete a document
collection.delete(ids=["doc_10"])
print(f"After delete: {collection.count()} documents")

## Key Takeaways

1. A **vector store** stores embeddings persistently so you don't re-compute them on every query
2. **ChromaDB** handles embedding automatically via an embedding function you provide
3. **Metadata** lets you filter results alongside semantic search
4. **CRUD operations** (add, get, update, delete) work similarly to a regular database

---

**Next:** Notebook 03 — Loading real documents and splitting them into chunks for effective retrieval.